# 7. 군집화(Clustering)
### CUAI 과제 (7-1 KMeans ~ 7-2 Cluster Evaluation)

> 군집화(Clustering) 는 레이블(정답) 없이 데이터 자체의 유사성을 기반으로  
> 비슷한 데이터끼리 같은 그룹으로 묶는 비지도 학습(Unsupervised Learning) 방법이다.  
> 정답이 없기 때문에 '얼마나 잘 군집됐는지'를 별도의 지표로 평가해야 한다.

---

## 7-1. K-Means 클러스터링

K-Means는 가장 대표적인 군집화 알고리즘.  
K개의 중심점(centroid) 을 기준으로 가장 가까운 데이터들을 같은 클러스터로 묶고,  
중심점을 반복적으로 이동하면서 군집을 최적화한다.

알고리즘 동작 순서:
1. K개의 초기 중심점을 랜덤 선택 (`init='k-means++'` 는 더 스마트한 초기화)
2. 각 데이터를 가장 가까운 중심점의 클러스터에 할당
3. 각 클러스터의 평균값으로 중심점 업데이트
4. 중심점 변화가 없을 때까지 2~3 반복

---
### K-Means를 이용한 붓꽃(Iris) 데이터 셋 Clustering

In [ ]:
from sklearn.preprocessing import scale
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
%matplotlib inline

iris = load_iris()
# 보다 편리한 데이터 Handling을 위해 DataFrame으로 변환
irisDF = pd.DataFrame(data=iris.data, columns=['sepal_length','sepal_width','petal_length','petal_width'])
irisDF.head(3)

In [ ]:
# KMeans 객체 생성 및 학습
# n_clusters=3 : 붓꽃 품종이 3개이므로 클러스터 수를 3으로 설정
# init='k-means++' : 초기 중심점을 스마트하게 선택해 수렴 속도 향상
# max_iter=300 : 최대 반복 횟수 (수렴 전 최대 300번까지 반복)
kmeans = KMeans(n_clusters=3, init='k-means++', max_iter=300, random_state=0)
kmeans.fit(irisDF)

In [ ]:
# labels_ : 각 데이터에 할당된 클러스터 번호 (0, 1, 2 중 하나)
# 실제 품종(target)과 비교해보면 얼마나 잘 나뉘었는지 확인 가능
print(kmeans.labels_)

In [ ]:
# target(실제 품종)과 cluster(KMeans 결과)를 함께 저장해서 비교
irisDF['target'] = iris.target
irisDF['cluster'] = kmeans.labels_

# groupby로 실제 품종별 클러스터 분포 확인
# → 완벽하게 일치하면 각 target마다 하나의 cluster에만 분포되어야 함
iris_result = irisDF.groupby(['target','cluster'])['sepal_length'].count()
print(iris_result)

In [ ]:
from sklearn.decomposition import PCA

# PCA로 4차원 데이터를 2차원으로 축소 → 시각화 가능하게 만들기
# PCA(Principal Component Analysis): 분산이 최대인 방향으로 차원 축소
pca = PCA(n_components=2)
pca_transformed = pca.fit_transform(iris.data)

irisDF['pca_x'] = pca_transformed[:, 0]  # 첫 번째 주성분
irisDF['pca_y'] = pca_transformed[:, 1]  # 두 번째 주성분
irisDF.head(3)

In [ ]:
# 클러스터별로 다른 마커를 사용해 2D 산점도로 시각화
# PCA로 축소된 2차원 좌표에서 클러스터가 얼마나 잘 구분되는지 확인

# 클러스터 값(0, 1, 2)에 해당하는 인덱스 추출
marker0_ind = irisDF[irisDF['cluster']==0].index
marker1_ind = irisDF[irisDF['cluster']==1].index
marker2_ind = irisDF[irisDF['cluster']==2].index

# 각 클러스터를 o, s, ^ 마커로 구분해서 scatter plot
plt.scatter(x=irisDF.loc[marker0_ind, 'pca_x'], y=irisDF.loc[marker0_ind, 'pca_y'], marker='o')
plt.scatter(x=irisDF.loc[marker1_ind, 'pca_x'], y=irisDF.loc[marker1_ind, 'pca_y'], marker='s')
plt.scatter(x=irisDF.loc[marker2_ind, 'pca_x'], y=irisDF.loc[marker2_ind, 'pca_y'], marker='^')

plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.title('3 Clusters Visualization by 2 PCA Components')
plt.show()

### Clustering 알고리즘 테스트를 위한 데이터 생성

`make_blobs()`로 군집이 명확한 인공 데이터를 만들어서 K-Means가 잘 동작하는지 확인.  
실제 데이터 적용 전에 알고리즘 자체의 동작 원리를 이해하는 데 유용하다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
%matplotlib inline

# make_blobs : 군집 중심(centers)을 기준으로 퍼진 인공 데이터 생성
# n_samples=200 : 총 200개 데이터
# n_features=2  : 2차원 (x, y) → 바로 시각화 가능
# centers=3      : 3개의 군집
# cluster_std=0.8: 군집 내 데이터 퍼짐 정도 (작을수록 뭉쳐있음)
X, y = make_blobs(n_samples=200, n_features=2, centers=3, cluster_std=0.8, random_state=0)
print(X.shape, y.shape)

# y target 값의 분포를 확인
unique, counts = np.unique(y, return_counts=True)
print(unique, counts)

In [ ]:
import pandas as pd

# X 데이터를 DataFrame으로 변환하고, 실제 정답(target) 컬럼 추가
clusterDF = pd.DataFrame(data=X, columns=['ftr1', 'ftr2'])
clusterDF['target'] = y
clusterDF.head(3)

In [ ]:
# 실제 정답(target) 기준으로 데이터 분포 시각화
# → K-Means 적용 전 원본 분포 확인
target_list = np.unique(y)
markers = ['o', 's', '^', 'P', 'D', 'H', 'x']  # 클러스터별 마커 종류

for target in target_list:
    target_cluster = clusterDF[clusterDF['target']==target]
    plt.scatter(x=target_cluster['ftr1'], y=target_cluster['ftr2'], edgecolor='k',
                marker=markers[target])

plt.show()

In [ ]:
# K-Means 클러스터링 적용 후 결과 시각화
# fit_predict() = fit() + predict() 한 번에 수행
kmeans = KMeans(n_clusters=3, init='k-means++', max_iter=200, random_state=0)
cluster_labels = kmeans.fit_predict(X)
clusterDF['kmeans_label'] = cluster_labels

# cluster_centers_ : 학습 후 각 클러스터의 최종 중심 좌표
centers = kmeans.cluster_centers_
unique_labels = np.unique(cluster_labels)
markers = ['o', 's', '^', 'P','D','H','x']

for label in unique_labels:
    label_cluster = clusterDF[clusterDF['kmeans_label']==label]
    center_x_y = centers[label]
    
    # 클러스터 데이터 포인트 scatter
    plt.scatter(x=label_cluster['ftr1'], y=label_cluster['ftr2'], edgecolor='k',
                marker=markers[label])
    
    # 중심점 시각화: 흰색 큰 마커 위에 클러스터 번호 표시
    plt.scatter(x=center_x_y[0], y=center_x_y[1], s=200, color='white',
                alpha=0.9, edgecolor='k', marker=markers[label])
    plt.scatter(x=center_x_y[0], y=center_x_y[1], s=70, color='k', edgecolor='k',
                marker='$%d$' % label)  # 숫자 마커로 클러스터 번호 표시

plt.show()

In [ ]:
# 실제 target과 K-Means가 예측한 kmeans_label의 분포 비교
# → 대부분 일치하면 군집이 잘 된 것
print(clusterDF.groupby('target')['kmeans_label'].value_counts())

---
## 7-2. 클러스터 평가 (Cluster Evaluation)

군집화는 정답이 없기 때문에 일반적인 정확도(accuracy)로 평가할 수 없다.  
대신 실루엣 계수(Silhouette Score) 를 사용하여 군집의 품질을 측정한다.

---
### 실루엣 분석 (Silhouette Analysis)

실루엣 계수는 각 데이터가 자신의 클러스터에 얼마나 잘 속해있는지를 나타낸다.

$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$

- a(i) : 같은 클러스터 내 다른 데이터들과의 평균 거리 (작을수록 좋음)
- b(i) : 가장 가까운 다른 클러스터까지의 평균 거리 (클수록 좋음)
- 범위: -1 ~ 1, 1에 가까울수록 잘 군집된 것

---
### 붓꽃(Iris) 데이터 셋을 이용한 클러스터 평가

In [ ]:
from sklearn.preprocessing import scale
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans
# 실루엣 분석 metric 값을 구하기 위한 API 추가
from sklearn.metrics import silhouette_samples, silhouette_score
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
%matplotlib inline

iris = load_iris()
feature_names = ['sepal_length','sepal_width','petal_length','petal_width']
irisDF = pd.DataFrame(data=iris.data, columns=feature_names)

# K-Means로 3개 클러스터로 군집화
kmeans = KMeans(n_clusters=3, init='k-means++', max_iter=300, random_state=0).fit(irisDF)
irisDF['cluster'] = kmeans.labels_

# silhouette_samples() : 각 데이터 포인트의 실루엣 계수를 개별로 반환
score_samples = silhouette_samples(iris.data, irisDF['cluster'])
print('silhouette_samples( ) return 값의 shape', score_samples.shape)

# 개별 실루엣 계수를 DataFrame에 추가
irisDF['silhouette_coeff'] = score_samples

# silhouette_score() : 전체 데이터의 평균 실루엣 계수 반환
average_score = silhouette_score(iris.data, irisDF['cluster'])
print('붓꽃 데이터셋 Silhouette Analysis Score:{0:.3f}'.format(average_score))

irisDF.head(3)

In [ ]:
# 클러스터별 평균 실루엣 계수 확인
# → 클러스터마다 점수가 균등하게 높아야 좋은 군집
# 특정 클러스터만 점수가 낮으면 해당 클러스터의 군집 품질이 나쁜 것
irisDF.groupby('cluster')['silhouette_coeff'].mean()

### 클러스터별 평균 실루엣 계수의 시각화를 통한 클러스터 개수 최적화 방법

최적의 K(클러스터 수)를 찾기 위해 여러 K값에 대해 실루엣 계수를 시각화하는 함수.  

**좋은 군집의 조건:**
1. 전체 평균 실루엣 계수가 높을 것
2. 모든 클러스터의 실루엣 계수가 평균과 비슷할 것 (특정 클러스터만 낮으면 안 됨)
3. 클러스터 크기가 너무 불균형하지 않을 것

In [ ]:
### 여러개의 클러스터링 갯수를 List로 입력 받아 각각의 실루엣 계수를 면적으로 시각화한 함수
def visualize_silhouette(cluster_lists, X_features): 
    
    from sklearn.datasets import make_blobs
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_samples, silhouette_score
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
    import math
    
    # 클러스터 수만큼 subplot 생성
    n_cols = len(cluster_lists)
    fig, axs = plt.subplots(figsize=(4*n_cols, 4), nrows=1, ncols=n_cols)
    
    # 각 클러스터 수에 대해 반복 수행
    for ind, n_cluster in enumerate(cluster_lists):
        
        # 해당 K로 KMeans 클러스터링 수행
        clusterer = KMeans(n_clusters=n_cluster, max_iter=500, random_state=0)
        cluster_labels = clusterer.fit_predict(X_features)
        
        # 전체 평균 실루엣 점수와 개별 실루엣 값 계산
        sil_avg = silhouette_score(X_features, cluster_labels)
        sil_values = silhouette_samples(X_features, cluster_labels)
        
        y_lower = 10
        axs[ind].set_title('Number of Cluster : '+ str(n_cluster)+'\n'
                          'Silhouette Score :' + str(round(sil_avg, 3)))
        axs[ind].set_xlabel("The silhouette coefficient values")
        axs[ind].set_ylabel("Cluster label")
        axs[ind].set_xlim([-0.1, 1])
        axs[ind].set_ylim([0, len(X_features) + (n_cluster + 1) * 10])
        axs[ind].set_yticks([])   # y축 눈금 제거 (클러스터 번호만 텍스트로 표시)
        axs[ind].set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1])
        
        # 클러스터별로 fill_betweenx() 형태의 막대 그래프 표현
        # → 막대가 넓을수록 해당 클러스터에 데이터가 많고, 오른쪽으로 길수록 실루엣 점수 높음
        for i in range(n_cluster):
            ith_cluster_sil_values = sil_values[cluster_labels==i]
            ith_cluster_sil_values.sort()  # 정렬해서 막대 모양이 고르게 표현되도록
            
            size_cluster_i = ith_cluster_sil_values.shape[0]
            y_upper = y_lower + size_cluster_i
            
            # 클러스터마다 다른 색상 지정
            color = cm.nipy_spectral(float(i) / n_cluster)
            axs[ind].fill_betweenx(np.arange(y_lower, y_upper), 0, ith_cluster_sil_values,
                                facecolor=color, edgecolor=color, alpha=0.7)
            axs[ind].text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))  # 클러스터 번호 표시
            y_lower = y_upper + 10
        
        # 빨간 점선: 전체 평균 실루엣 점수 기준선
        # 모든 클러스터의 막대가 이 선을 넘어야 좋은 군집
        axs[ind].axvline(x=sil_avg, color="red", linestyle="--")

In [ ]:
# make_blobs로 4개 클러스터 중심의 500개 2차원 데이터 생성
from sklearn.datasets import make_blobs
X, y = make_blobs(n_samples=500, n_features=2, centers=4, cluster_std=1,
                  center_box=(-10.0, 10.0), shuffle=True, random_state=1)

# K를 2, 3, 4, 5로 바꿔가며 실루엣 계수 시각화
# → 실제 클러스터 수(4)일 때 실루엣 점수가 가장 높고 균등하게 나와야 함
visualize_silhouette([2, 3, 4, 5], X)

In [ ]:
# 붓꽃 데이터에도 동일하게 적용
# → 실제 품종이 3개이지만 가장 좋은 K가 몇인지 실루엣으로 확인
from sklearn.datasets import load_iris

iris = load_iris()
visualize_silhouette([2, 3, 4, 5], iris.data)